# 03 - Training the FNO surrogate

§11.2 steps 7 and 8: train the operator, and run the physics-loss ablation.

Before the loop starts, three things are established on a single batch, because each has a
failure mode that a converging training curve would hide:

- the parameter count is what §4.4 claims, and the radial mask is actually masking;
- each of the four loss terms is the size it is supposed to be relative to the others, and
  the physics term has a **measurable floor** -- the residual of the *true* field on the
  network grid, which no prediction can beat;
- `balance_alpha` returns something sane when measured on the projection head.

**Runtime.** `EPOCHS = 300` on 2000 samples at `BATCH_SIZE = 16` x 4 frequencies is roughly
8-16 hours on an A100, and the ablation doubles it. Set `SMOKE = True` to run 3 epochs and
confirm the plumbing end to end in a few minutes; the headless launcher is the way to do the
real run:

```
modal run modal_app.py::train_both
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

In [ ]:
SMOKE = True          # <-- set False for the real 300-epoch run

EPOCHS = 3 if SMOKE else cfg.EPOCHS
WORKERS = 2           # make_loader's docstring: drop to 2 on smaller CPU allocations
print(f"{'SMOKE TEST' if SMOKE else 'FULL RUN'}: {EPOCHS} epochs, "
      f"{WORKERS} loader workers")
assert paths["train"].exists() and paths["val"].exists(), \
    "run notebook 02 first -- there is no dataset to train on"

## The operator

Four Fourier blocks at `d_v = 32`, `KMAX = 28` retained modes, radial truncation, no
normalisation layers, and the final block linear.

`effective_params` and `allocated_params` differ, and the difference is the point of radial
truncation. The weight tensors are `kmax x kmax` boxes, but the mask keeps only
`|k| <= kmax` on the integer lattice, so about `pi/4` of the box survives -- a 21% saving
with no loss of resolved bandwidth, because the corner modes of the box are at
`|k| = sqrt(2) kmax`, beyond anything the physics puts energy in. The masked coefficients are
zeroed at construction and re-masked on every forward pass, so `effective_params` is the
number of real parameters that can actually move, and it is the number quoted.

In [ ]:
from src.models import fno2d

model = fno2d.build("primary").to(DEV)
print(model.summary())
print(f"\nconfig.total_params (independent arithmetic): {cfg.total_params():,}")
print(f"model.effective_params:                        {model.effective_params():,}")
print(f"model.allocated_params:                        {model.allocated_params():,}")
print(f"radial truncation saves "
      f"{1 - model.effective_params()/model.allocated_params():.1%} of the spectral "
      f"weights")
print(f"\nfinal block linear: act={model.blocks[-1].act} "
      f"(the others: {[b.act for b in model.blocks[:-1]]})")

In [ ]:
sc = model.blocks[0].spectral
m1 = _np(sc.m1)[0, 0]
m2 = _np(sc.m2)[0, 0]

fig, ax = plt.subplots(1, 3, figsize=(10.2, 3.0))
ax[0].imshow(m1, origin="lower", cmap="Greys_r", interpolation="nearest")
ax[0].set(title=f"mask, k_y >= 0 block\n{int(m1.sum())} of {m1.size} modes kept",
          xlabel="k_x", ylabel="k_y")
ax[1].imshow(m2, origin="lower", cmap="Greys_r", interpolation="nearest")
ax[1].set(title="mask, k_y < 0 block\n(row iy is k_y = -(kmax - iy))", xlabel="k_x")

kk = np.hypot(*np.meshgrid(np.arange(sc.kmax), np.arange(sc.kmax), indexing="ij"))
w = np.abs(_np(sc.w1)[0, 0])
ax[2].semilogy(kk[m1 > 0].ravel(), np.maximum(w[m1 > 0].ravel(), 1e-12), ".",
               ms=2, alpha=0.5, label="kept")
if (m1 == 0).any():
    ax[2].semilogy(kk[m1 == 0].ravel(), np.maximum(w[m1 == 0].ravel(), 1e-12), ".",
                   ms=2, c="C3", alpha=0.6, label="masked (exactly 0)")
ax[2].axvline(sc.kmax, ls="--", c="0.4", lw=1.0, label=f"|k| = KMAX = {sc.kmax}")
ax[2].set(xlabel="|k|", ylabel="|w| at init", title="masked weights are zero, not small")
ax[2].legend(fontsize=7.5)
for a_ in ax[:2]:
    a_.grid(False)
fig.tight_layout()
savefig(fig, "03_radial_mask.png")
plt.show()

## The four loss terms on one batch

    L = L_field + gamma L_H1 + beta L_meas + alpha L_phys

The table below evaluates the same batch four ways, adding one term at a time. Two rows
matter more than the rest:

- **prediction = 0** (the untrained network outputs almost nothing). `L_field` is then 1.0
  by construction, and `L_phys` is the residual of `u_inc` alone in the presence of the
  void -- which is exactly the scattering source term the network is being asked to cancel.
  A near-zero value here would mean the physics term cannot see the defect at all.
- **prediction = target** (the labels themselves). `L_field`, `L_H1` and `L_meas` are
  exactly 0; `L_phys` is *not*, and its value is the floor. The residual is evaluated on the
  `128^2` network grid at half the solver's resolution, so the 4th-order stencil's own
  dispersion error and the 1.5-cell coefficient transition at the void boundary set a floor
  the network cannot go below no matter how right it is. Every `L_phys` printed during
  training should be read against this number, not against zero.

That floor is also the argument for balancing `alpha` instead of fixing it.

In [ ]:
from src import features as feat
from src import losses as L
from src import training
from src.data.dataset import WaveDataset, batch_to_model, make_loader, to_device

ds = WaveDataset(str(paths["train"]), train=True)
ds.set_epoch(0)
batch = to_device(next(iter(make_loader(ds, batch_size=8, num_workers=0))), DEV)
x, y = batch_to_model(batch)
recv = training.receivers_tensor(DEV)
ctx = L.make_context(batch["chi"], batch["nu"], batch["freqs"], batch["src_idx"])
u_inc = feat.flatten_freq(batch["u_inc"])
print(f"batch: x {tuple(x.shape)}  y {tuple(y.shape)}  "
      f"ctx.weight {tuple(ctx.weight.shape)}")

In [ ]:
gen = torch.Generator().manual_seed(cfg.SEED)

def terms_of(pred, **kw):
    return L.compute(pred, y, recv_yx=recv, ctx=ctx, u_inc=u_inc,
                     generator=torch.Generator().manual_seed(cfg.SEED), **kw)

with torch.no_grad():
    pred0 = model(x)
    combos = [
        ("field only",          dict(gamma=0.0, beta=0.0, alpha=0.0)),
        ("+ H1",                dict(gamma=cfg.GAMMA_H1, beta=0.0, alpha=0.0)),
        ("+ ring",              dict(gamma=cfg.GAMMA_H1, beta=cfg.BETA_MEAS, alpha=0.0)),
        ("+ physics (default)", dict(gamma=cfg.GAMMA_H1, beta=cfg.BETA_MEAS,
                                     alpha=cfg.ALPHA_PHYS)),
    ]
    rows = []
    for label, kw in combos:
        t = terms_of(pred0, **kw)
        rows.append((label, f"{float(t.total):.4f}", f"{float(t.field):.4f}",
                     f"{float(t.h1):.4f}", f"{float(t.meas):.4f}",
                     f"{float(t.phys):.4f}", f"{t.alpha:.1e}"))
    t_zero = terms_of(torch.zeros_like(pred0), **combos[-1][1])
    t_true = terms_of(y.clone(), **combos[-1][1])

table(rows, ["untrained model", "total", "field", "H1", "meas", "phys", "alpha"])
print()
table([("prediction = 0", f"{float(t_zero.field):.4f}", f"{float(t_zero.h1):.4f}",
        f"{float(t_zero.meas):.4f}", f"{float(t_zero.phys):.6f}"),
       ("prediction = target", f"{float(t_true.field):.4f}", f"{float(t_true.h1):.4f}",
        f"{float(t_true.meas):.4f}", f"{float(t_true.phys):.6f}")],
      ["reference point", "field", "H1", "meas", "phys"])

PHYS_FLOOR = float(t_true.phys)
print(f"\nL_phys floor (true field on the 128^2 grid): {PHYS_FLOOR:.6f}")
print(f"L_phys with no scattered field at all:      {float(t_zero.phys):.6f}")
print(f"ratio {float(t_zero.phys)/max(PHYS_FLOOR, 1e-30):.1f}x -- the physics term can "
      f"see the defect")
print(f"\ngamma = {cfg.GAMMA_H1} not 1.0: at 16 points per lambda_p the discrete gradient")
print(f"amplifies the high-|k| error by ~k dx, so the H1 term's natural scale is already")
print(f"{float(t_zero.h1)/max(float(t_zero.field), 1e-30):.2f}x the field term's.")

## `balance_alpha` on the projection head

`alpha` such that `alpha ||grad L_phys|| = 0.1 ||grad L_data||`, measured on
`model.project.parameters()` -- the two 1x1 convolutions of the projection head, and the
last place the two gradients are still the same kind of quantity. Measuring on the whole
network would average over the lifting layers, where the physics term's gradient has been
diluted by the depth it passed through and the ratio stops meaning anything.

It costs two extra backward passes on a retained graph, which is why the training loop calls
it every 200 steps and not every step, and calls it *before* `opt.step()`: the retained
graph's saved activations belong to the current parameters, so balancing after the step would
measure a ratio at a point the network is no longer at.

In [ ]:
pred = model(x)
t = L.compute(pred, y, recv_yx=recv, ctx=ctx, u_inc=u_inc, alpha=cfg.ALPHA_PHYS,
              generator=torch.Generator().manual_seed(cfg.SEED))
l_data = t.field + cfg.GAMMA_H1 * t.h1 + cfg.BETA_MEAS * t.meas
balance_params = [p for p in model.project.parameters() if p.requires_grad]
print(f"balancing on {len(balance_params)} tensors, "
      f"{sum(p.numel() for p in balance_params):,} parameters "
      f"(the projection head)")

a_fresh = L.balance_alpha(l_data, t.phys, balance_params)
a_ema = L.balance_alpha(l_data, t.phys, balance_params, alpha_prev=cfg.ALPHA_PHYS)
print(f"\nalpha from the gradient-norm ratio : {a_fresh:.4e}")
print(f"alpha EMA-smoothed from {cfg.ALPHA_PHYS:.0e}     : {a_ema:.4e}")
print(f"config default ALPHA_PHYS         : {cfg.ALPHA_PHYS:.4e}")
print(f"clamp                             : (1e-5, 1.0)")

# whole-network alpha, to show why the projection head is the right place to measure
a_all = L.balance_alpha(l_data, t.phys,
                        [p for p in model.parameters() if p.requires_grad],
                        alpha_prev=None)
print(f"\nsame ratio measured on ALL parameters: {a_all:.4e} "
      f"({a_all/max(a_fresh, 1e-30):.2f}x different)")
ds.close()
del pred, t, l_data

## The two arms of the ablation

One flag: `alpha=None` disables the physics term entirely and takes the identical code path
otherwise, so the two runs cannot differ in the data pipeline, the schedule, the seed or
anything else. §11.2 step 8 asks whether the physics term buys anything; the honest way to
answer that is two runs that differ in one argument.

`num_workers=2` rather than the default 4: each worker holds its own HDF5 read buffer and
the bottleneck is chunk decompression of `128^2` complex planes, so on a container with a
handful of CPUs more workers means more memory and no more throughput.

In [ ]:
ARMS = {
    "nophys": dict(alpha=None),
    "full":   dict(alpha=cfg.ALPHA_PHYS),
}
hists = {}

for name, kw in ARMS.items():
    out = E.checkpoints / name
    hp = out / "history.json"
    if hp.exists() and not SMOKE:
        hists[name] = json.loads(hp.read_text())
        print(f"{name}: history.json exists, {len(hists[name]['train'])} epochs "
              f"-- skipping")
        continue
    print(f"\n{'='*72}\n=== arm '{name}': {kw}, {EPOCHS} epochs -> {out}\n{'='*72}")
    m = fno2d.build("primary")
    t0 = time.perf_counter()
    hists[name] = training.train(
        m, str(paths["train"]), str(paths["val"]), out_dir=str(out), device=DEV,
        epochs=EPOCHS, num_workers=WORKERS, progress=tqdm, **kw)
    print(f"arm '{name}' finished in {(time.perf_counter()-t0)/60:.1f} min")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(11.5, 6.0))
for name, h in hists.items():
    ep = np.arange(len(h["train"]))
    ax[0, 0].semilogy(ep, [r["total"] for r in h["train"]], lw=1.0, label=name)
    ax[0, 1].semilogy(ep, [r["rel_l2"] for r in h["val"]], lw=1.0, label=name)
    ax[0, 2].semilogy(ep, [r["ring"] for r in h["val"]], lw=1.0, label=name)
    ax[1, 0].semilogy(ep, [r["phase"] for r in h["val"]], lw=1.0, label=name)
    ax[1, 1].semilogy(ep, [max(r["phys"], 1e-12) for r in h["train"]], lw=1.0,
                      label=f"{name}: L_phys")
    ax[1, 2].semilogy(ep, [max(a, 1e-12) for a in h["alpha"]], lw=1.0, label=name)

ax[0, 0].set(xlabel="epoch", ylabel="train total loss", title="training loss")
ax[0, 1].set(xlabel="epoch", ylabel="val rel-L2", title="field error")
ax[0, 1].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[0, 2].set(xlabel="epoch", ylabel="val rel-L2 on the ring", title="receiver ring")
ax[1, 0].set(xlabel="epoch", ylabel="phase error (periods)", title="arrival error")
ax[1, 0].axhline(cfg.GATE_ARRIVAL_PERIODS, ls="--", c="C3", lw=1.0)
ax[1, 1].axhline(PHYS_FLOOR, ls=":", c="0.35", lw=1.2, label="floor (true field)")
ax[1, 1].set(xlabel="epoch", ylabel="L_phys", title="physics residual vs its floor")
ax[1, 2].set(xlabel="epoch", ylabel="alpha", title="balanced alpha")
for a_ in ax.ravel():
    a_.legend(fontsize=7)
fig.tight_layout()
savefig(fig, "03_training_curves.png")
plt.show()

## Gates

`rel_l2 < 5%` on the field and arrival error `< 0.05` periods, both from §11.3. The arrival
error is measured as receiver *phase* error divided by 2 pi, not by synthesising a time trace:
the band is 0.66-1.34 f_c, so a synthesised trace has a time resolution of about 1.5 periods
and could not resolve a 0.05-period error even in principle. Receivers whose true scattered
amplitude is below 5% of the row maximum are excluded, because phase is meaningless where
there is no signal and a shadowed receiver would contribute a uniformly distributed error of
0.25 periods on average.

In [ ]:
from src.data.dataset import WaveDataset as WD

va = WD(str(paths["val"]), train=False)
vl = make_loader(va, batch_size=4, shuffle=False, num_workers=0)

evals = {}
for name in ARMS:
    ck = E.checkpoints / name / "best.pt"
    if not ck.exists():
        print(f"{name}: no checkpoint")
        continue
    m, meta = training.load(ck, device=DEV)
    evals[name] = training.evaluate(m, vl, DEV, per_freq=True)
    print(f"\n--- {name}  (best epoch {meta['epoch']}, alpha {meta['alpha']})")
    print(evals[name])
va.close()

if len(evals) == 2:
    a, b = evals["nophys"], evals["full"]
    print(f"\nablation (§11.2 step 8)")
    table([("field rel-L2", f"{a.rel_l2:.4f}", f"{b.rel_l2:.4f}",
            f"{(a.rel_l2-b.rel_l2)/max(a.rel_l2,1e-30):+.1%}"),
           ("ring rel-L2", f"{a.ring_rel_l2:.4f}", f"{b.ring_rel_l2:.4f}",
            f"{(a.ring_rel_l2-b.ring_rel_l2)/max(a.ring_rel_l2,1e-30):+.1%}"),
           ("phase (periods)", f"{a.phase_periods:.4f}", f"{b.phase_periods:.4f}",
            f"{(a.phase_periods-b.phase_periods)/max(a.phase_periods,1e-30):+.1%}")],
          ["metric", "no physics", "full", "physics gains"])
    if SMOKE:
        print("\n(SMOKE: 3 epochs.  This comparison means nothing yet -- the physics\n"
              "term acts on generalisation, which needs the full schedule.)")

## The checkpoint carries its architecture

A `state_dict` alone cannot reconstruct this model. `d_v` and `kmax` set the shape of every
spectral weight, so loading a `primary` checkpoint into a `small` model raises a shape error
-- but loading a checkpoint trained with `radial=False` into `radial=True` does **not**: the
shapes match and the operator is silently different. `save` therefore stores the constructor
arguments alongside the weights, and the round-trip below asserts bit-identical outputs.

In [ ]:
ck = E.checkpoints / "full" / "best.pt"
if ck.exists():
    m, meta = training.load(ck, device=DEV)
    print("stored arch:", meta["arch"])
    print("stored val :", meta["val"])
    with torch.no_grad():
        p1 = m(x)
        training.save(m, E.checkpoints / "roundtrip.pt", epoch=meta["epoch"])
        m2, _ = training.load(E.checkpoints / "roundtrip.pt", device=DEV)
        p2 = m2(x)
    print(f"\nround-trip max |difference|: {float((p1-p2).abs().max()):.3e}")
    assert torch.equal(p1, p2), "checkpoint round-trip is not exact"
    print("PASS  bit-identical after save/load")
    (E.checkpoints / "roundtrip.pt").unlink()
else:
    print("no full/best.pt yet")

## Capacity variants

The three variants of §4.4 differ only in `d_v` and `KMAX`. `tiny` at `KMAX = 16` is worth
noting: `band_in_modes()` says the top of the band at the worst Poisson ratio needs mode index
~18, so `tiny` **truncates inside the physics** -- it cannot represent the shortest shear
wave in the band, and its error should concentrate at the top of the band rather than being
uniformly worse. That is a prediction, and notebook 04's error-vs-|k| figure is where it gets
checked.

In [ ]:
k_need = fno2d.band_in_modes()
rows = []
for v, kw in cfg.VARIANTS.items():
    m = fno2d.build(v)
    rows.append((v, kw["d_v"], kw["kmax"], f"{m.effective_params():,}",
                 f"{m.allocated_params():,}",
                 "yes" if kw["kmax"] >= k_need else "NO -- truncates the band"))
    del m
table(rows, ["variant", "d_v", "kmax", "effective", "allocated", "covers the band?"])
print(f"\nband_in_modes() = {k_need:.1f} at nu = {min(cfg.NU_LIST)}, "
      f"f = {max(cfg.FREQS):.2f} f_c")

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "smoke": SMOKE, "epochs": EPOCHS,
    "params": {v: fno2d.build(v).effective_params() for v in cfg.VARIANTS},
    "phys_floor_true_field": PHYS_FLOOR,
    "phys_zero_prediction": float(t_zero.phys),
    "alpha_balanced_projection": a_fresh,
    "alpha_balanced_all_params": a_all,
    "arms": {k: dict(rel_l2=v.rel_l2, ring=v.ring_rel_l2, phase=v.phase_periods,
                     per_freq=v.per_freq, gates=v.gates())
             for k, v in evals.items()},
}
dump(record, "03_train_fno.json")
print("\nCheckpoints:")
for p in sorted(E.checkpoints.glob("*/best.pt")):
    print(f"  {p}  {p.stat().st_size/1e6:.1f} MB")
print("\nNotebook 04 evaluates checkpoints/full/best.pt as a forward operator.")